In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from google.colab import drive
from skimage.measure import regionprops, label
from tqdm import tqdm
import matplotlib.pyplot as plt

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define path to your dataset folder
root_path = '/content/drive/MyDrive/soyabean'  # change this if your dataset is in a different folder name
folders = ['black', 'brown', 'unfit']

# Step 3: Create a function to extract morphological features
def extract_features(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply thresholding
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Invert if needed (object must be white, background black)
    if np.sum(thresh == 255) < np.sum(thresh == 0):
        thresh = cv2.bitwise_not(thresh)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return None  # No seed found

    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)

    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = float(w) / h if h != 0 else 0

    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = float(area) / hull_area if hull_area != 0 else 0

    roundness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0

    # Eccentricity using skimage
    labeled = label(thresh)
    props = regionprops(labeled)
    eccentricity = props[0].eccentricity if props else 0

    return [area, perimeter, aspect_ratio, eccentricity, solidity, roundness]

# Step 4: Loop through each folder and extract features
for folder in folders:
    folder_path = os.path.join(root_path, folder)
    data = []
    image_names = []

    print(f"Processing folder: {folder}")
    for filename in tqdm(os.listdir(folder_path)):
        if filename.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
            img_path = os.path.join(folder_path, filename)
            features = extract_features(img_path)
            if features:
                data.append(features)
                image_names.append(filename)

    df = pd.DataFrame(data, columns=[
        'Area', 'Perimeter', 'Aspect_Ratio', 'Eccentricity', 'Solidity', 'Roundness'
    ])
    df.insert(0, 'Image_Name', image_names)

    # Save CSV to Google Drive
    csv_path = f"/content/{folder.replace(' ', '_')}_features.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing folder: black


100%|██████████| 200/200 [00:07<00:00, 25.15it/s]


Saved: /content/black_features.csv
Processing folder: brown


100%|██████████| 200/200 [00:10<00:00, 19.44it/s]


Saved: /content/brown_features.csv
Processing folder: unfit


100%|██████████| 210/210 [00:15<00:00, 13.27it/s]

Saved: /content/unfit_features.csv


In [ ]:
import time
from google.colab import files

# Download black
files.download('/content/black_features.csv')
time.sleep(2)  # Wait 2 seconds

# Download brown
files.download('/content/brown_features.csv')
time.sleep(2)

# Download unfit
files.download('/content/unfit_features.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>